# G04: ReAct + Calculator (Qwen3-4B), chuyên đề Mũ-Logarit, 3 dạng: D78, D68, D56

Notebook chạy **lần lượt 3 dạng** mà Qwen3-4B zero-shot chưa đạt 100%, nạp model **một lần**. Mỗi dạng chạy đủ
90 câu và ghi một file `<mã dạng>_react_calculator_full90.csv` trong `/kaggle/working`, cuối cùng ghi `summary_G04.csv`.

| Dạng | Nội dung | Zero-shot đúng | Số lệnh Calculator tối đa | Độ dài prompt (ký tự) |
|---|---|---|---|---|
| D78 | tìm giá trị nhỏ nhất của $P=x^2+y^2+\alpha x+\beta y$ với $x,y\ge0$, $2x+y\cdot4^{x+y-1}\ge3$ | 62/90 | 9 | 19,844 |
| D68 | tìm $a+2b$ biết $\log_{pa+qb+1}((ra)^2+(sb)^2+1)+\log_{tab+1}(pa+qb+1)=2$ | 65/90 | 8 | 18,991 |
| D56 | đếm số nguyên $x$ thỏa $\log_p\frac{x^2-A}{q^m}<\log_q\frac{x^2-A}{p^m}$ | 66/90 | 8 | 19,025 |

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool `Calculator` (sympy, chính xác tuyệt đối, có bộ nhớ biến)
theo vòng lặp ReAct thật: `Thought → Action → Action Input → Observation → ...`

Prompt mỗi dạng gồm 5 phần giống bộ Số phức và dạng D79: `PART_TOOL` + `PART_KIENTHUC` (dùng chung) + `PART_HUONGGIAI`
(hướng giải theo lời giải mẫu của câu gốc) + `PART_FEWSHOT` (các `Observation` là kết quả chạy thật bằng chính backend) + `PART_NHIEMVU`.
Tham số sinh: `temperature=0.3, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2, max_new_tokens=9216`, ép mở đầu `<think>\nThought:`.

## Trước khi chạy

1. Gắn Kaggle Dataset chứa `plan_solve_prompts_mulogarit_merged.json` (đã dùng cho zero-shot). Notebook tự dò đường dẫn bằng `glob`.
2. Bật GPU (khuyến nghị 2×T4), chọn **Save & Run All**.
3. Muốn thử nhanh vài câu: đặt `RUN_STT = [1, 2, 3]` và/hoặc `RUN_TYPES` ở ô cấu hình.


In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec torchaudio
import vllm; print('vLLM:', vllm.__version__)


In [ ]:
import os, re, json, time, glob
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

GROUP = 'G04'
RUN_TYPES = ['D78', 'D68', 'D56']      # the types of this notebook, in running order
DATA_FILENAME = 'plan_solve_prompts_mulogarit_merged.json'
# The dataset folder name under /kaggle/input differs between accounts, so
# search for the file instead of hardcoding a path.
_matches = glob.glob(f'/kaggle/input/**/{DATA_FILENAME}', recursive=True)
assert _matches, f'{DATA_FILENAME} not found under /kaggle/input. Add the dataset to the notebook first.'
DATA_PATH = _matches[0]

RUN_STT = None            # None = all 90 questions; for a quick test use e.g. [1, 2, 3]

MODEL = 'Qwen/Qwen3-4B'
# Same decoding settings as the Complex Numbers ReAct + Calculator runs.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_NEW_TOKENS = 9216

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    _all = json.load(f)
ALL_RECORDS = {ma: _all[ma] for ma in RUN_TYPES}
print({ma: len(v) for ma, v in ALL_RECORDS.items()})

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)


In [ ]:
# =====================================================================
# PROMPTS. PART_TOOL and PART_KIENTHUC are shared by every problem type of the
# chapter; PART_HUONGGIAI, PART_FEWSHOT and PART_NHIEMVU are specific to each type.
# TYPE_PROMPTS[type]['template'] is the five parts joined; {de_bai} is the question.
# =====================================================================

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: rate = log(6)/log(5)` → `Observation: rate = log(6)/log(5)`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: big = N(101**rate - 101, 20)`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Power is `**` (e.g. `x**2`, `101**rate`), not `^`. Square root is `sqrt(...)`; never the `√` symbol.
- Plain fractions are already exact: writing `7/2` gives exactly seven halves, never a decimal. You do NOT need any special function for fractions.
- Logarithms: `log(u)` is the NATURAL logarithm (write `log`, never `ln`). A logarithm to another base is written with the change of base formula, `log(u)/log(base)`, or equivalently `log(u, base)`. The exponential function is `exp(u)` and Euler's number is `E`.
- Absolute value is `Abs(u)`. Rounding: `floor(u)` is the greatest integer not above `u`, and `ceiling(u)` is the least integer not below `u`. `Min(u, v)` and `Max(u, v)` give the smaller and the larger of two numbers.
- Exact results stay exact (for example `log(8)/log(2)` becomes `3`). When you need a decimal value, for instance to see between which integers a number lies, write `N(expression, 20)`: this is the Calculator computing a decimal for you, not you approximating.
- To solve an equation, write `solve(expression, unknown)`, where the expression is the left side of the equation once everything is moved to one side so that it equals 0. It returns the list of solutions. A letter that the method tells you to use as an unknown (for example `x`, `y`, `t`, or the letters `a` and `b` of a formula) may be used inside `solve`, `diff`, `expand`, `factor` and `subs` without being assigned first; never store a number under that letter. Other tools: `expand(...)`, `factor(...)`, `diff(f, x)`, and `expr.subs(x, value)` to substitute.
- Some methods use these extra tools, always in exactly the form the method shows: `nsolve(expression, unknown, start)` finds a root numerically starting the search at `start`; `real_roots(polynomial)` lists the real roots of a polynomial exactly; `Sum(term, (k, first, last)).doit()` adds the terms exactly; `Count(condition, (n, low, high))` counts the integers `n` from `low` to `high` that make the condition true (it can be nested, and the counting letters need no earlier assignment); `Piecewise((value, condition), (other_value, True))` chooses a value by a condition; `And(...)` joins conditions; `oo` is infinity.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately. To compare two numbers by size, write `left < right`, `left <= right`, `left > right` or `left >= right`; the Calculator decides it with 50 digits of precision and answers `True` or `False`. Never write `==`.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation solving mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non real ones; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`) and discard it immediately if not. For a root confirmed real, use `re(root)`, not the raw value. Any comparison built on numeric roots should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root, in particular the final count and matching it against the answer options, stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''

PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF EXPONENTIALS AND LOGARITHMS**

1. **Powers**: for $a>0$ and real $m,n$: $a^m a^n=a^{m+n}$, $\dfrac{a^m}{a^n}=a^{m-n}$, $(a^m)^n=a^{mn}$, $(ab)^n=a^n b^n$, $a^{-n}=\dfrac{1}{a^n}$, $a^0=1$, and $a^{m/n}=\sqrt[n]{a^m}$.
2. **Exponential function** $y=a^x$ with $a>0$, $a\ne1$: it is defined for every real $x$, and $a^x>0$ always (it never equals 0 and is never negative). It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$.
3. **Exponential equations**: $a^{f}=a^{g}\iff f=g$. For a number $c$: $a^{f}=c$ has a solution only when $c>0$, and then $f=\log_a c$.
4. **Exponential inequalities** (same base): if $a>1$, then $a^{f}<a^{g}\iff f<g$ (direction kept). If $0<a<1$, then $a^{f}<a^{g}\iff f>g$ (direction reversed). In particular the sign of $a^{u}-a^{v}$ equals the sign of $u-v$ when $a>1$.
5. **Substitution $t=a^x$**: because $a^x>0$, the new variable must satisfy $t>0$. Then $a^{2x}=t^2$ and $a^{x+k}=a^k\,t$.
6. **Definition of the logarithm**: for $a>0$, $a\ne1$, $b>0$: $\log_a b=c\iff a^c=b$. The base must be positive and not equal to 1, and the argument must be strictly positive.
7. **Basic values**: $\log_a 1=0$, $\log_a a=1$, $\log_a a^c=c$, and $a^{\log_a b}=b$ for $b>0$.
8. **Product, quotient, power** (for $x,y>0$): $\log_a(xy)=\log_a x+\log_a y$, $\log_a\dfrac{x}{y}=\log_a x-\log_a y$, $\log_a x^{n}=n\log_a x$. For an even power of a number $x\ne0$ that may be negative: $\log_a x^{2k}=2k\log_a|x|$. Also $\log_{a^m}x=\dfrac{1}{m}\log_a x$ and $\log_a\sqrt[n]{x}=\dfrac{1}{n}\log_a x$.
9. **Change of base**: $\log_a b=\dfrac{\log_c b}{\log_c a}$ for any base $c>0$, $c\ne1$. Consequences: $\log_a b=\dfrac{1}{\log_b a}$ (for $b\ne1$) and $\log_a b\cdot\log_b c=\log_a c$.
10. **Natural and common logarithm**: $\ln x=\log_e x$ and $\log x=\log_{10}x$. In particular $\log_a b=\dfrac{\ln b}{\ln a}$.
11. **Exchanging base and exponent**: $a^{\log_b c}=c^{\log_b a}$ for $a,c>0$ and $b>0$, $b\ne1$. Reason: taking $\log_b$ of each side gives $\log_b c\cdot\log_b a$ on both sides, and two positive numbers with the same logarithm are equal.
12. **Logarithmic function** $y=\log_a x$: its domain is $x>0$. It is strictly increasing when $a>1$ and strictly decreasing when $0<a<1$. If $a>1$: $\log_a u<\log_a v\iff 0<u<v$ (direction kept). If $0<a<1$: $\log_a u<\log_a v\iff u>v>0$ (direction reversed). In both cases the arguments must be positive first.
13. **Logarithmic equations**: first state the domain (every argument $>0$). Then $\log_a f=\log_a g\iff f=g$, and $\log_a f=c\iff f=a^c$. A solution is accepted only if it satisfies the domain.
14. **Logarithmic inequalities with a constant**: if $a>1$: $\log_a f>c\iff f>a^c$ and $\log_a f<c\iff 0<f<a^c$. If $0<a<1$: $\log_a f>c\iff 0<f<a^c$ and $\log_a f<c\iff f>a^c$.
15. **Substitution $t=\log_a x$**: here $x>0$ and $t$ can be any real number. Then $\log_a^2 x=t^2$ and $x=a^t$.
16. **Always find the domain first**: every logarithm argument $>0$, every base positive and not 1, every denominator $\ne0$, every expression under a square root $\ge0$. Any value found later is accepted only if it satisfies this domain.
17. **Sign of a difference of two logarithms** (base $a>1$, arguments $u,v>0$): the sign of $\log_a u-\log_a v$ equals the sign of $u-v$. In particular $\log_a u-\log_a v=0\iff u=v$.
18. **Products and square roots**: $A\cdot B=0$ means $A=0$ or $B=0$. For $A\cdot\sqrt{B}=0$: either $A=0$ (and then $B\ge0$ must hold so that the square root exists), or $B=0$. The sign of a product $A\cdot B$ is decided by the signs of its two factors: $A\cdot B\le 0$ means one factor is $\ge0$ and the other is $\le0$.
19. **Derivatives**: $(a^x)'=a^x\ln a$, $(e^x)'=e^x$, $(a^u)'=u'\,a^u\ln a$. Also $(\log_a x)'=\dfrac{1}{x\ln a}$, $(\ln x)'=\dfrac{1}{x}$, $(\log_a u)'=\dfrac{u'}{u\ln a}$, and for a quotient $\left(\dfrac{u}{v}\right)'=\dfrac{u'v-uv'}{v^2}$.
20. **Derivative and monotonicity**: if $f'(x)>0$ on an interval, $f$ is strictly increasing there; if $f'(x)<0$, it is strictly decreasing. A strictly increasing function satisfies $f(u)<f(v)\iff u<v$ and $f(u)=f(v)\iff u=v$, so the equation $f=k$ has at most one solution.
21. **A power function with exponent above 1**: if $\alpha>1$ then $f(t)=t^{\alpha}-t$ has $f'(t)=\alpha t^{\alpha-1}-1$, and for $t\ge1$ we get $t^{\alpha-1}\ge1$, so $f'(t)\ge\alpha-1>0$: $f$ is strictly increasing on $[1,+\infty)$.
22. **Comparing the bases**: if $a>b>1$ then $\log_b a=\dfrac{\ln a}{\ln b}>1$.
23. **Integers**: a sum or difference of integers is an integer, and a positive integer is at least 1. Different integer values of $t$ give different values of $y=t-x$ for a fixed integer $x$, so counting the integers $y$ is the same as counting the integers $t$.
24. **Counting integers**: for real numbers $p<q$, the number of integers strictly between them (open interval $(p,q)$) is $(\lceil q\rceil-1)-(\lfloor p\rfloor+1)+1=\lceil q\rceil-\lfloor p\rfloor-1$. The number of integers in the closed interval $[p,q]$ is $\lfloor q\rfloor-\lceil p\rceil+1$. Here $\lfloor u\rfloor$ is the greatest integer not above $u$ and $\lceil u\rceil$ is the least integer not below $u$. The integers $1,2,\dots,T$ are exactly $T$ numbers.
25. **Quadratic equation** $ax^2+bx+c=0$ ($a\ne0$): $\Delta=b^2-4ac$. If $\Delta>0$ there are two distinct real roots $x_{1,2}=\dfrac{-b\pm\sqrt{\Delta}}{2a}$; if $\Delta=0$ one double root; if $\Delta<0$ no real root. Viète: $x_1+x_2=-\dfrac{b}{a}$ and $x_1x_2=\dfrac{c}{a}$.
26. **Quadratic inequality**: for $a>0$ and $\Delta>0$ with roots $x_1<x_2$: $ax^2+bx+c<0\iff x_1<x<x_2$, and $ax^2+bx+c>0\iff x<x_1$ or $x>x_2$. The two roots themselves are included only for the non strict signs $\le$ and $\ge$.
27. **Two distinct positive roots**: the quadratic $at^2+bt+c=0$ has two distinct positive roots if and only if $\Delta>0$, $x_1+x_2>0$ and $x_1x_2>0$.
28. **Squares and square roots**: $\sqrt{u^2}=|u|$. For $k>0$: $x^2<k\iff-\sqrt{k}<x<\sqrt{k}$ and $x^2>k\iff x<-\sqrt{k}$ or $x>\sqrt{k}$.
29. **Inequality of the arithmetic and geometric means**: for $u,v\ge0$: $u+v\ge2\sqrt{uv}$, with equality if and only if $u=v$.
30. **Bunyakovsky inequality** (also called Cauchy and Schwarz): for real numbers $p,q,x,y$: $(px+qy)^2\le(p^2+q^2)(x^2+y^2)$, with equality if and only if $(p,q)$ and $(x,y)$ are proportional.
31. **Values of a continuous monotone function**: if $f$ is continuous and strictly increasing on $[p,q]$, then on the open interval $(p,q)$ it takes every value strictly between $f(p)$ and $f(q)$ exactly once, and no other value. So $f(x)=k$ has a solution in $(p,q)$ if and only if $f(p)<k<f(q)$. For a strictly decreasing $f$ the roles of $f(p)$ and $f(q)$ swap.
32. **Behaviour at the ends** (base $a>1$): $a^x\to0^+$ when $x\to-\infty$ and $a^x\to+\infty$ when $x\to+\infty$; $\ln x\to-\infty$ when $x\to0^+$ and $\ln x\to+\infty$ when $x\to+\infty$. A function that goes from $-\infty$ up to a local maximum, down to a local minimum, then up to $+\infty$ crosses the $x$ axis 3 times if the maximum is positive and the minimum is negative, and once otherwise.
33. **Quadratic in $t=a^x>0$**: the roots of $t^2+Bt+D=0$ are accepted only if they are positive. If $D<0$ the roots have opposite signs, and $t^2+Bt+D>0$ with $t>0$ means $t$ above the larger root. A negative sign in front of a number is part of the number: for example in $t^2-12t+35$ we have $B=-12$.
'''

TYPE_PROMPTS = {}

# ---------------------------------------------------------------- D78
D78_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: non negative reals $x,y$ satisfy $2x+y\cdot4^{x+y-1}\ge3$, and we want the minimum of $P=x^2+y^2+\alpha x+\beta y$ (with $\alpha,\beta>0$). Write the condition as $y\cdot4^{x+y-1}\ge3-2x$.

**Case 1: $x\ge\frac32$.** Then $3-2x\le0$, so the condition holds for every $y\ge0$. Since $y^2+\beta y\ge0$, we get $P\ge x^2+\alpha x$, and $x^2+\alpha x$ is increasing for $x\ge\frac32$ (its derivative $2x+\alpha$ is positive). So in this case $P\ge\frac94+\frac32\alpha$, reached at $x=\frac32,\ y=0$.

**Case 2: $x<\frac32$.** Now $3-2x>0$, so the condition forces $y>0$, and we can divide by $y$ and take logarithms to the base 4, then convert to the base 2:
$$4^{x+y-1}\ge\frac{3-2x}{y}\iff x+y-1\ge\tfrac12\log_2\frac{3-2x}{y}\iff 2x+2y-2\ge\log_2(3-2x)-\log_2y.$$
Rearranging gives $2y+\log_2(2y)\ge(3-2x)+\log_2(3-2x)$. The function $f(t)=t+\log_2t$ is strictly increasing for $t>0$, because $f'(t)=1+\dfrac{1}{t\ln2}>0$. Hence the condition is equivalent to
$$2y\ge3-2x,\qquad\text{that is }y\ge\tfrac32-x.$$
Since $P$ increases with $y\ge0$, its smallest value for a given $x$ is reached at $y=\frac32-x$ (this point satisfies the original condition with equality, because then $x+y=\frac32$ and $4^{1/2}=2$). Substituting $y=\frac32-x$:
$$P\ge Q(x)=x^2+\left(\tfrac32-x\right)^2+\alpha x+\beta\left(\tfrac32-x\right)=2x^2+(\alpha-\beta-3)x+\tfrac94+\tfrac32\beta.$$

**Minimising $Q$.** $Q$ is a quadratic with positive leading coefficient. Its vertex is at $x_v=\dfrac{3+\beta-\alpha}{4}$. The allowed $x$ in this case is $0\le x\le\frac32$ (at $x=\frac32$ Case 2 meets Case 1), so the minimum is attained at the vertex when $x_v$ lies in $[0;\tfrac32]$, and otherwise at the nearer end. In Case 1 the minimum $\frac94+\frac32\alpha$ equals $Q\left(\frac32\right)$, so it is already covered. Therefore
$$P_{\min}=Q\big(\operatorname{clamp}(x_v,\,0,\,\tfrac32)\big).$$
The value is compared with the four options.
'''

D78_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Xét các số thực không âm $x,y$ thỏa mãn $2x+y\cdot4^{x+y-1}\ge3$. Giá trị nhỏ nhất của $P=x^2+y^2+23x+25y$ bằng

A. $P_{\min}=\dfrac{301}{8}$.
B. $P_{\min}=\dfrac{293}{8}$.
C. $P_{\min}=\dfrac{285}{8}$.
D. $P_{\min}=\dfrac{293}{4}$.

<think>
Thought: I read the problem. The expression is $P=x^2+y^2+23x+25y$, so the coefficient of $x$ is 23. I store it as `coef_x`.
Action: Calculator
Action Input: coef_x = 23
Observation: coef_x = 23

Thought: The coefficient of $y$ is 25. I store it as `coef_y`.
Action: Calculator
Action Input: coef_y = 25
Observation: coef_y = 25

Thought: The vertex of $Q(x)$ is $x_v=\dfrac{3+\beta-\alpha}{4}$, with $\alpha=$ `coef_x` and $\beta=$ `coef_y`. I store it as `x_vertex`.
Action: Calculator
Action Input: x_vertex = (3 + coef_y - coef_x)/4
Observation: x_vertex = 5/4

Thought: The allowed $x$ in Case 2 is from 0 to $\frac32$, so I clamp the vertex into this range: the larger of 0 and the smaller of $\frac32$ and `x_vertex`. I store it as `x_best`.
Action: Calculator
Action Input: x_best = Max(0, Min(3/2, x_vertex))
Observation: x_best = 5/4

Thought: Now $Q(x)=2x^2+(\alpha-\beta-3)x+\frac94+\frac32\beta$ at `x_best`. This is the minimum of $P$.
Action: Calculator
Action Input: answer = 2*x_best**2 + (coef_x - coef_y - 3)*x_best + 9/4 + 3*coef_y/2
Observation: answer = 293/8

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $P_{\min}=\dfrac{301}{8}$, B: $P_{\min}=\dfrac{293}{8}$, C: $P_{\min}=\dfrac{285}{8}$, D: $P_{\min}=\dfrac{293}{4}$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $P_{\min}=\dfrac{301}{8}$; in Calculator syntax this is `301/8`.
Action: Calculator
Action Input: test_A = Eq(answer, 301/8)
Observation: test_A = False

Thought: Option B is $P_{\min}=\dfrac{293}{8}$; in Calculator syntax this is `293/8`.
Action: Calculator
Action Input: test_B = Eq(answer, 293/8)
Observation: test_B = True

Thought: Option C is $P_{\min}=\dfrac{285}{8}$; in Calculator syntax this is `285/8`.
Action: Calculator
Action Input: test_C = Eq(answer, 285/8)
Observation: test_C = False

Thought: Option D is $P_{\min}=\dfrac{293}{4}$; in Calculator syntax this is `293/4`.
Action: Calculator
Action Input: test_D = Eq(answer, 293/4)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $P_{\min}=\dfrac{301}{8}$ gives False, B: $P_{\min}=\dfrac{293}{8}$ gives True, C: $P_{\min}=\dfrac{285}{8}$ gives False, D: $P_{\min}=\dfrac{293}{4}$ gives False. Exactly one option matches, and it is B.

</think>
Final Answer: \boxed{B}
'''

D78_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the coefficient $\alpha$ of $x$ and the coefficient $\beta$ of $y$ in $P=x^2+y^2+\alpha x+\beta y$ (a coefficient that is not written is 1).
[Step 2] Store `coef_x = <alpha>`, then `coef_y = <beta>`.
[Step 3] Store `x_vertex = (3 + coef_y - coef_x)/4`.
[Step 4] Store `x_best = Max(0, Min(3/2, x_vertex))`.
[Step 5] Store `answer = 2*x_best**2 + (coef_x - coef_y - 3)*x_best + 9/4 + 3*coef_y/2`.
[Step 6] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D78'] = {
    'template': PART_TOOL + PART_KIENTHUC + D78_HUONGGIAI + D78_FEWSHOT + D78_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


# ---------------------------------------------------------------- D68
D68_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: positive reals $a,b$ satisfy $\log_{U}\big((ra)^2+(sb)^2+1\big)+\log_{V}(U)=2$, where $U=pa+qb+1$, $V=tab+1$ and $t=2rs$. We are asked for $a+2b$. (In the original problem the first argument is written $25a^2+b^2+1=(5a)^2+(b)^2+1$.)

**All bases and arguments exceed 1.** Since $a,b>0$, we have $U>1$, $V>1$ and $(ra)^2+(sb)^2+1>1$, so every logarithm is positive and defined.

**First inequality (AM-GM).** For non negative numbers, $(ra)^2+(sb)^2\ge2\cdot ra\cdot sb=2rs\,ab=t\,ab$, so $(ra)^2+(sb)^2+1\ge tab+1=V$. The logarithm to the base $U>1$ is increasing, so
$$\log_U\big((ra)^2+(sb)^2+1\big)\ge\log_U V.$$

**Second inequality (AM-GM again).** Hence the left side of the equation is at least $\log_U V+\log_V U$. Both terms are positive and their product is $\log_U V\cdot\log_V U=1$, so by AM-GM their sum is at least $2\sqrt{1}=2$. The equation says the sum equals exactly 2, so both inequalities must be equalities.

**The equality conditions.** The first one holds iff $ra=sb$, that is $b=\dfrac rs\,a$. The second holds iff $\log_U V=\log_V U$, and since their product is 1 this means $\log_U V=1$, that is $V=U$:
$$tab+1=pa+qb+1\iff t\,ab=pa+qb.$$

**Solving.** Put $\rho=\dfrac rs$, so $b=\rho a$. Then $t\rho a^2=(p+q\rho)\,a$, and because $a>0$,
$$a=\frac{p+q\rho}{t\rho},\qquad b=\rho a.$$
The requested number is $a+2b$. (Check on the original problem: $r=5$, $s=1$, $p=10$, $q=3$, $t=10$ give $\rho=5$, $a=\frac12$, $b=\frac52$, $a+2b=\frac{11}{2}$.) The value is compared with the four options.
'''

D68_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Cho $a>0,b>0$ thỏa mãn
$$\log_{12a+5b+1}((3a)^2+(b)^2+1)+\log_{6ab+1}(12a+5b+1)=2.$$
Giá trị của $a+2b$ bằng

A. $\dfrac{19}{2}$.
B. $\dfrac{21}{2}$.
C. $\dfrac{23}{2}$.
D. $21$.

<think>
Thought: I read the problem. The first argument is $(3a)^2+(b)^2+1$, so $r=3$ and $s=1$ (a bare $b$ means 1). The equality condition $ra=sb$ gives $b=\rho a$ with $\rho=\dfrac rs$. I store `ratio`.
Action: Calculator
Action Input: ratio = 3/1
Observation: ratio = 3

Thought: The base of the first logarithm is $12a+5b+1$, so $p=12$ and $q=5$. The coefficient in $6ab+1$ is $t=6$ (which equals $2rs$). From $t\rho a^2=(p+q\rho)a$ with $a>0$ I get $a=\dfrac{p+q\rho}{t\rho}$. I store it as `a_value`.
Action: Calculator
Action Input: a_value = (12 + 5*ratio)/(6*ratio)
Observation: a_value = 3/2

Thought: Then $b=\rho a$. I store it as `b_value`.
Action: Calculator
Action Input: b_value = ratio*a_value
Observation: b_value = 9/2

Thought: The requested number is $a+2b$.
Action: Calculator
Action Input: answer = a_value + 2*b_value
Observation: answer = 21/2

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $\dfrac{19}{2}$, B: $\dfrac{21}{2}$, C: $\dfrac{23}{2}$, D: $21$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $\dfrac{19}{2}$; in Calculator syntax this is `19/2`.
Action: Calculator
Action Input: test_A = Eq(answer, 19/2)
Observation: test_A = False

Thought: Option B is $\dfrac{21}{2}$; in Calculator syntax this is `21/2`.
Action: Calculator
Action Input: test_B = Eq(answer, 21/2)
Observation: test_B = True

Thought: Option C is $\dfrac{23}{2}$; in Calculator syntax this is `23/2`.
Action: Calculator
Action Input: test_C = Eq(answer, 23/2)
Observation: test_C = False

Thought: Option D is $21$.
Action: Calculator
Action Input: test_D = Eq(answer, 21)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $\dfrac{19}{2}$ gives False, B: $\dfrac{21}{2}$ gives True, C: $\dfrac{23}{2}$ gives False, D: $21$ gives False. Exactly one option matches, and it is B.

</think>
Final Answer: \boxed{B}
'''

D68_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: from the base $pa+qb+1$ of the first logarithm, the numbers $p$ and $q$; from the first argument $(ra)^2+(sb)^2+1$, the numbers $r$ and $s$ (a bare $a$ or $b$ means 1); and from the base $tab+1$ of the second logarithm, the number $t$.
[Step 2] Store `ratio = <r>/<s>`.
[Step 3] Store `a_value = (<p> + <q>*ratio)/(<t>*ratio)`.
[Step 4] Store `b_value = ratio*a_value`.
[Step 5] Store `answer = a_value + 2*b_value`.
[Step 6] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D68'] = {
    'template': PART_TOOL + PART_KIENTHUC + D68_HUONGGIAI + D68_FEWSHOT + D68_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


# ---------------------------------------------------------------- D56
D56_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: the problem gives $\log_p\dfrac{x^2-A}{B}<\log_q\dfrac{x^2-A}{C}$ with $B=q^m$, $C=p^m$ and $q>p>1$, and asks how many integers $x$ satisfy it.

**Domain.** Both arguments are positive exactly when $u=x^2-A>0$.

**Splitting the quotients.** Put $u=x^2-A>0$. Then
$$\log_p u-\log_p B<\log_q u-\log_q C,\qquad \log_p B=m\log_p q,\quad \log_q C=m\log_q p.$$
Let $s=\log_p q$; since $q>p>1$ we have $s>1$, and $\log_q p=\dfrac1s$, $\log_q u=\dfrac{\log_p u}{s}$. The inequality becomes
$$\log_p u-ms<\frac{\log_p u}{s}-\frac ms\iff\log_p u\left(1-\frac1s\right)<m\left(s-\frac1s\right).$$

**Collecting the terms in $\log_p u$.** Since $1-\frac1s=\frac{s-1}{s}>0$ and $s-\frac1s=\frac{(s-1)(s+1)}{s}$, dividing by the positive number $\frac{s-1}{s}$ gives
$$\log_p u<m(s+1)=m\log_p q+m=\log_p\big(q^mp^m\big)=\log_p\big((pq)^m\big).$$
The logarithm to the base $p>1$ is increasing, so $u<(pq)^m=q^m\cdot p^m=B\cdot C$.

**The integers.** Together with the domain, $0<x^2-A<B\cdot C$, that is $A<x^2<A+B\cdot C$, that is $\sqrt A<|x|<\sqrt{A+BC}$. The values $x$ and $-x$ both work, so we count the positive integers $n$ with $\sqrt A<n<\sqrt{A+BC}$ and double the count. The largest such $n$ is $\lceil\sqrt{A+BC}\rceil-1$ (the ceiling takes care of the strict inequality even if $\sqrt{A+BC}$ is an integer), and the smallest is $\lfloor\sqrt A\rfloor+1$ (again strict). So the number of positive integers is $\big(\lceil\sqrt{A+BC}\rceil-1\big)-\lfloor\sqrt A\rfloor$, and the answer is twice that. It is compared with the four options.
'''

D56_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Có bao nhiêu số nguyên $x$ thỏa mãn $\log_{3}\dfrac{x^2-25}{1024} < \log_{4}\dfrac{x^2-25}{243}$?

A. $985$.
B. $986$.
C. $984$.
D. $493$.

<think>
Thought: I read the problem. The constant under $x^2$ is $A=25$. The two denominators are $B=1024$ (under the first logarithm) and $C=243$ (under the second). The method says $x^2-A<B\cdot C$, so I multiply the two denominators and store the product as `limit_value`.
Action: Calculator
Action Input: limit_value = 1024*243
Observation: limit_value = 248832

Thought: The positive integers $n$ with $n^2<A+BC$ are $1,\dots,\lceil\sqrt{A+BC}\rceil-1$ (the ceiling handles the strict inequality). I store the largest one as `n_high`.
Action: Calculator
Action Input: n_high = ceiling(sqrt(25 + limit_value)) - 1
Observation: n_high = 498

Thought: The positive integers $n$ with $n^2\le A$ are those up to $\lfloor\sqrt A\rfloor$, and they must be excluded because $n^2>A$ is required. I store this number as `n_low`.
Action: Calculator
Action Input: n_low = floor(sqrt(25))
Observation: n_low = 5

Thought: The positive integers $n$ with $\sqrt A<n<\sqrt{A+BC}$ number `n_high - n_low`, and each gives the two values $\pm n$, so the answer is twice that.
Action: Calculator
Action Input: answer = 2*(n_high - n_low)
Observation: answer = 986

Thought: The computation is done and its results are stored. Before testing anything, I copy all four options exactly as they are written in the problem, side by side: A: $985$, B: $986$, C: $984$, D: $493$. I will write each one in Calculator syntax, test them one call per option, and read each result next to the option I copied here.

Thought: Option A is $985$.
Action: Calculator
Action Input: test_A = Eq(answer, 985)
Observation: test_A = False

Thought: Option B is $986$.
Action: Calculator
Action Input: test_B = Eq(answer, 986)
Observation: test_B = True

Thought: Option C is $984$.
Action: Calculator
Action Input: test_C = Eq(answer, 984)
Observation: test_C = False

Thought: Option D is $493$.
Action: Calculator
Action Input: test_D = Eq(answer, 493)
Observation: test_D = False

Thought: Lining up what I copied with the stored results: A: $985$ gives False, B: $986$ gives True, C: $984$ gives False, D: $493$ gives False. Exactly one option matches, and it is B.

</think>
Final Answer: \boxed{B}
'''

D56_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Read the problem and write down, without computing: the number $A$ in $x^2-A$ (the same in both logarithms), the denominator $B$ under the first logarithm and the denominator $C$ under the second logarithm.
[Step 2] Store `limit_value = <B>*<C>`, writing the two denominators from the problem.
[Step 3] Store `n_high = ceiling(sqrt(<A> + limit_value)) - 1`.
[Step 4] Store `n_low = floor(sqrt(<A>))`.
[Step 5] Store `answer = 2*(n_high - n_low)`.
[Step 6] Copy all four options side by side in one Thought, and write each one in Calculator syntax (a fraction `\dfrac{p}{q}` becomes `p/q`, `\sqrt{n}` becomes `sqrt(n)`, a power `u^{k}` becomes `u**k`). Then test every option in order with `test_A = Eq(answer, <option A>)`, then `test_B`, `test_C`, `test_D` in the same way. Finally line up each copied option next to its stored result; exactly one of them is `True`. Do not pick the letter by eye.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''

TYPE_PROMPTS['D56'] = {
    'template': PART_TOOL + PART_KIENTHUC + D56_HUONGGIAI + D56_FEWSHOT + D56_NHIEMVU,
    'free_unknowns': [],
    'max_tool_calls': 25,
}


In [ ]:
# The context window must hold the longest prompt plus the whole generation budget.
PROMPT_TOKENS = 0
for ma, spec in TYPE_PROMPTS.items():
    longest = max(ALL_RECORDS[ma], key=lambda r: len(r['de_bai_mcq']))
    n_tok = len(tok(spec['template'].replace('{de_bai}', longest['de_bai_mcq'])).input_ids)
    print(f'{ma}: prompt {n_tok} tokens')
    PROMPT_TOKENS = max(PROMPT_TOKENS, n_tok)
MAX_MODEL_LEN = ((PROMPT_TOKENS + MAX_NEW_TOKENS + 1500) // 1024 + 1) * 1024
print(f'Longest prompt: {PROMPT_TOKENS} tokens -> MAX_MODEL_LEN={MAX_MODEL_LEN}')

N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'GPUs: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('Ready.')


In [ ]:
# =====================================================================
# BACKEND: the Calculator tool (SymPy with variable memory)
# =====================================================================
import re

import sympy as sp

ASSIGN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Names that already mean something to the Calculator (functions such as N, Min, floor and
# constants such as E, I, S). Storing a value under one of them would silently replace it,
# for example "N = 94" would make the later call N(...) fail for the rest of the question.
RESERVED_NAMES = {n for n in dir(sp) if not n.startswith('_')}
RELATIONAL = (sp.StrictLessThan, sp.LessThan, sp.StrictGreaterThan, sp.GreaterThan, sp.Unequality)


class Count(sp.Expr):
    """Count(condition, (n, low, high)): how many integers n from low to high make the condition true.

    The condition is decided exactly, one integer at a time (like a calculator table). It may itself
    contain another Count over a second variable; the inner Count is evaluated only after the outer
    variable has been replaced by a number, so nothing is ever expanded symbolically."""

    def __new__(cls, condition, limits, low=None, high=None):
        if low is None:
            var, low, high = limits
        else:
            var = limits
        obj = sp.Expr.__new__(cls, condition, var, sp.sympify(low), sp.sympify(high))
        if getattr(condition, 'free_symbols', set()) - {var}:
            return obj                      # still depends on another variable: wait
        return obj.doit()

    @property
    def free_symbols(self):
        condition, var, low, high = self.args
        return (condition.free_symbols - {var}) | low.free_symbols | high.free_symbols

    def doit(self, **hints):
        condition, var, low, high = self.args
        total = 0
        for value in range(int(low), int(high) + 1):
            decided = condition.subs(var, sp.Integer(value))
            decided = decided.doit() if hasattr(decided, 'doit') else decided
            if decided is sp.true or decided is True:
                total += 1
            elif not (decided is sp.false or decided is False):
                raise ValueError(f'the condition could not be decided for {var} = {value}: {decided}')
        return sp.Integer(total)


EXTRA_NAMES = {'Count': Count}
RESERVED_NAMES = RESERVED_NAMES | set(EXTRA_NAMES)


class Calculator:
    """SymPy calculator with variable memory (one fresh instance per question)."""

    def __init__(self, free_unknowns=()):
        self.free_unknowns = set(free_unknowns)
        # Free unknowns are registered up front as REAL symbols; letting
        # sympify create them lazily would make them generic complex symbols.
        self.ns = {name: sp.Symbol(name, real=True) for name in self.free_unknowns}

    def _undefined_names(self, expr):
        free = getattr(expr, 'free_symbols', set())
        missing = {str(s) for s in free} - self.free_unknowns
        if not missing:
            return None
        unknown = ', '.join(sorted(missing))
        in_memory = ', '.join(sorted(self.ns)) or '(nothing stored yet)'
        allowed = ', '.join(sorted(self.free_unknowns)) or '(none)'
        return (f'undefined name(s): {unknown}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {in_memory}. Declared free unknowns '
                f'(allowed without prior assignment): {allowed}. Either compute '
                'and store that name first (with "name = expression"), or '
                'rewrite the expression without it.')

    def _unknown_functions(self, parsed):
        """Names used as functions that the Calculator does not know (ceil, log10, int, ...).

        SymPy accepts any unknown name followed by parentheses and silently keeps it as an
        unevaluated symbol, so "count = ceil(115.36) - 1" would be stored as if it were a
        result. Refuse it with a message that says what to write instead."""
        items = parsed if isinstance(parsed, (list, tuple, sp.FiniteSet)) else [parsed]
        names = set()
        for item in items:
            for node in getattr(item, 'atoms', lambda *a: set())(sp.core.function.AppliedUndef):
                names.add(type(node).__name__)
        if not names:
            return None
        return (f'unknown function(s): {", ".join(sorted(names))}. The Calculator does not know '
                'these names, so nothing was computed. Use only: log, exp, sqrt, floor, '
                'ceiling, Abs, Min, Max, N, solve, expand, factor, diff, nroots, Eq. '
                'For example: round up with ceiling(u), round down with floor(u), a logarithm '
                'to base b as log(u)/log(b).')

    def _decide_relational(self, rel):
        """True or False for an inequality between constant expressions."""
        err = self._undefined_names(rel.lhs - rel.rhs)
        if err:
            return None, err
        if rel.lhs.free_symbols or rel.rhs.free_symbols:
            return None, ('cannot decide an inequality that still contains an '
                          'unknown. Substitute numbers first.')
        diff = sp.N(rel.lhs - rel.rhs, 50)
        if abs(diff) < sp.Float('1e-40'):
            diff = sp.Integer(0)
        table = {sp.StrictLessThan: diff < 0, sp.LessThan: diff <= 0,
                 sp.StrictGreaterThan: diff > 0, sp.GreaterThan: diff >= 0,
                 sp.Unequality: diff != 0}
        return ('True' if table[type(rel)] else 'False'), None

    def _boolean_result(self, name, truth):
        """Format a True/False answer; store it when it was assigned to a name."""
        text = 'True' if truth else 'False'
        if name:
            self.ns[name] = sp.true if truth else sp.false
            return f'{name} = {text}', None
        return text, None

    def evaluate(self, expression):
        """Return (result_text, error_text). Exactly one of them is None."""
        s = expression.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        name = None
        m = ASSIGN_RE.match(s)
        # "x = Eq(...)" is an assignment; a bare "Eq(a, b)" has no top level "=".
        if m and not s.lstrip().startswith('Eq('):
            name, s = m.group(1), m.group(2)
            if name in self.free_unknowns:
                return None, (f'"{name}" is a declared unknown used by solve(...) and cannot be '
                              'a variable name. Choose another name and store the value again.')
            if name in RESERVED_NAMES:
                return None, (f'"{name}" cannot be used as a variable name because it is the name '
                              'of a Calculator function or constant (for example N(expression, '
                              'digits) computes a decimal value). Choose another name, such as a '
                              'descriptive word, and store the value again.')

        try:
            parsed = sp.sympify(s, locals={**EXTRA_NAMES, **self.ns})
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        err = self._unknown_functions(parsed)
        if err:
            return None, err

        try:
            if isinstance(parsed, sp.Equality):
                err = self._undefined_names(parsed.lhs - parsed.rhs)
                if err:
                    return None, err
                same = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return self._boolean_result(name, same)

            if isinstance(parsed, RELATIONAL):
                text, err = self._decide_relational(parsed)
                if err:
                    return None, err
                return self._boolean_result(name, text == 'True')

            if isinstance(parsed, (sp.logic.boolalg.BooleanTrue, sp.logic.boolalg.BooleanFalse)):
                return self._boolean_result(name, bool(parsed))

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Result of solve(...): a list of solutions. Check each element
                # separately instead of simplifying the list as a whole.
                items = list(parsed)
                for item in items:
                    err = self._undefined_names(item)
                    if err:
                        return None, err
                if name:
                    self.ns[name] = items
                    return f'{name} = {items}', None
                return str(items), None

            value = sp.expand(sp.simplify(parsed))
            err = self._undefined_names(value)
            if err:
                return None, err
            if name:
                self.ns[name] = value
                return f'{name} = {value}', None
            return str(value), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# BATCHED ReAct LOOP (one function call = one problem type, all its questions):
# one vLLM call per round for every active question, then the Calculator
# runs separately for each question.
# =====================================================================
STOP_STR = 'Observation:'
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class QuestionState:
    """ReAct state of ONE question (own variable memory, own token budget)."""

    def __init__(self, rec, chat_prompt, free_unknowns):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.calculator = Calculator(free_unknowns=free_unknowns)
        # Forced prefix: the model must continue from "Thought:" right after
        # <think>, it cannot start with free prose.
        self.full_text = '<think>\nThought:'
        self.total_tokens = 0
        self.n_calls = 0
        self.tool_log = []
        self.done = False
        self.stop_reason = ''
        self.final_round = False          # tool limit reached: force the Final Answer


def extract_letter(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*(?:\\text\{\s*)?([A-D])\b',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


def run_type(ma_dang):
    spec = TYPE_PROMPTS[ma_dang]
    free_unknowns = set(spec['free_unknowns'])
    max_tool_calls = spec['max_tool_calls']
    records = ALL_RECORDS[ma_dang]
    if RUN_STT is not None:
        records = [r for r in records if r['STT'] in RUN_STT]
    out_path = f'/kaggle/working/{ma_dang.lower()}_react_calculator_full90.csv'
    print('=' * 72)
    print(f'{ma_dang}: {len(records)} questions')
    print('=' * 72)

    states = []
    for r in records:
        prompt_text = spec['template'].replace('{de_bai}', r['de_bai_mcq'])
        chat = tok.apply_chat_template([{'role': 'user', 'content': prompt_text}],
                                       tokenize=False, add_generation_prompt=True, enable_thinking=True)
        states.append(QuestionState(r, chat, free_unknowns))

    t0 = time.time()
    round_no = 0
    while True:
        active = [s for s in states if not s.done]
        if not active:
            break
        round_no += 1

        batch = []
        for s in active:
            model_input = s.chat_prompt + s.full_text
            room = MAX_MODEL_LEN - len(tok(model_input).input_ids) - 8
            n_new = min(MAX_NEW_TOKENS - s.total_tokens, room)
            if n_new <= 0:
                s.done = True
                s.stop_reason = 'out_of_context' if room <= 0 else 'out_of_token_budget'
                continue
            stop = None if s.final_round else [STOP_STR]
            batch.append((s, model_input, SamplingParams(
                temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                presence_penalty=PRESENCE_PENALTY, max_tokens=n_new,
                seed=SEED, stop=stop)))

        if not batch:
            break

        outs = llm.generate([p for _, p, _ in batch],
                            [sp_ for _, _, sp_ in batch], use_tqdm=False)

        calls_this_round = 0
        for (s, _, _), out in zip(batch, outs):
            o = out.outputs[0]
            s.full_text += o.text
            s.total_tokens += len(o.token_ids)

            if s.final_round:
                s.done = True
                s.stop_reason = 'tool_limit_reached'
                continue

            if o.stop_reason != STOP_STR:
                s.done = True
                s.stop_reason = ('model_finished' if o.finish_reason == 'stop'
                                 else 'out_of_tokens')
                continue

            s.n_calls += 1
            calls_this_round += 1
            match = None
            for mm in ACTION_INPUT_RE.finditer(o.text):
                match = mm
            expression = match.group(1).strip() if match else ''

            if not expression:
                observation = ('ERROR: no "Action Input:" line found. Write a Thought, '
                               'then "Action: Calculator", then "Action Input: <expression>".')
                s.tool_log.append(('(no Action Input)', observation))
            else:
                result, error = s.calculator.evaluate(expression)
                if error:
                    observation = (f'ERROR: {error}. Fix the syntax and try again '
                                   '(sqrt() for roots, ** for powers, log() for logarithms, '
                                   'Eq(a, b) to compare).')
                    s.tool_log.append((expression, f'ERROR: {error}'))
                else:
                    observation = result
                    s.tool_log.append((expression, result))

            s.full_text += f'{STOP_STR} {observation}\n'

            if s.n_calls >= max_tool_calls:
                s.full_text += ('\n[SYSTEM: tool-call limit reached, give your Final '
                                'Answer now, with no further Action.]\n')
                s.final_round = True

        print(f'  Round {round_no:2d}: {len(batch):2d} questions generating, '
              f'{calls_this_round:2d} tool calls, '
              f'{sum(1 for s in states if not s.done):2d} still active '
              f'({time.time()-t0:.0f}s)')

    print(f'Finished after {round_no} rounds, {(time.time()-t0)/60:.1f} minutes.')

    rows = []
    for s in states:
        chosen = extract_letter(s.full_text)
        n_err = sum(1 for _, obs in s.tool_log if str(obs).startswith('ERROR'))
        rows.append({
            'STT': s.rec['STT'],
            'ma_dang': s.rec.get('ma_dang', ma_dang),
            'dap_an_dung': s.rec['dap_an_letter'],
            'model_chon': chosen,
            'DUNG': chosen == s.rec['dap_an_letter'],
            'token': s.total_tokens,
            'tool_calls': s.n_calls,
            'syntax_errors': n_err,
            'stop_reason': s.stop_reason,
            'tool_log': ' | '.join(f'{ex} -> {obs}' for ex, obs in s.tool_log),
            'full_text': s.full_text,
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding='utf-8-sig')

    n_right = int(df['DUNG'].sum())
    n_none = int((df['model_chon'] == '').sum())
    print(f"RESULT {ma_dang}: {n_right} / {len(df)} correct ({df['DUNG'].mean()*100:.1f}%)   "
          f"wrong={len(df) - n_right - n_none}   undetermined={n_none}")
    print(f"Average tool calls per question : {df['tool_calls'].mean():.1f}   "
          f"Total syntax errors: {df['syntax_errors'].sum()} "
          f"({df['syntax_errors'].sum() / max(1, df['tool_calls'].sum()) * 100:.1f}%)   "
          f"Questions with NO tool call: {(df['tool_calls'] == 0).sum()}   "
          f"Average tokens: {df['token'].mean():.0f}")
    print(df['stop_reason'].value_counts().to_string())
    if (~df['DUNG']).any():
        print('--- WRONG OR UNDETERMINED QUESTIONS ---')
        print(df[~df['DUNG']][['STT', 'dap_an_dung', 'model_chon', 'token',
                               'tool_calls', 'syntax_errors', 'stop_reason']].to_string(index=False))
    else:
        print(f'*** ALL {len(df)} QUESTIONS CORRECT ***')
    print('Saved:', out_path)
    return df


# =====================================================================
# RUN EVERY TYPE OF THIS NOTEBOOK, ONE AFTER THE OTHER
# =====================================================================
summary_rows = []
for ma in RUN_TYPES:
    d_ = run_type(ma)
    summary_rows.append({'ma_dang': ma, 'tong': len(d_), 'dung': int(d_['DUNG'].sum()),
                         'ti_le_dung': round(float(d_['DUNG'].mean()) * 100, 1),
                         'tool_calls_tb': round(float(d_['tool_calls'].mean()), 1),
                         'syntax_errors': int(d_['syntax_errors'].sum()),
                         'token_tb': round(float(d_['token'].mean()))})
    pd.DataFrame(summary_rows).to_csv(f'/kaggle/working/summary_{GROUP}.csv', index=False, encoding='utf-8-sig')

print()
print('=' * 72)
print('SUMMARY')
print('=' * 72)
print(pd.DataFrame(summary_rows).to_string(index=False))
